In [ ]:
# USGS metadata cleaning – dates & keys
# -------------------------------------------------------------
# This script is meant to run in a Jupyter notebook. You can
# paste cells one-by-one, or run as a .py script. It:
#   - Loads the CSV (no edits to the original file)
#   - Creates a unique key id_page_number
#   - Builds cleaned coordinates (best-available lat/long)
#   - Standardizes water_type into fixed categories (with review column)
#   - Parses dates_of_recording into year_start / year_end
#   - Enforces the rule: ignore any year > 1980 and any
#     "present/current year/to date" open-ends
#   - Exports a non-destructive cleaned copy + a small
#     manual-review file for messy dates and water types
# -------------------------------------------------------------

# %% Imports
import os
import re
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

# %% User paths (Windows)
DATA_DIR = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data"
INPUT_CSV = os.path.join(DATA_DIR, "cleaned_metadata_final.csv")
# Safe outputs alongside input (non-destructive)
OUTPUT_CSV = os.path.join(DATA_DIR, "cleaned_metadata_with_years.csv")
REVIEW_CSV = os.path.join(DATA_DIR, "dates_needs_review.csv")
WATER_TYPE_REVIEW_CSV = os.path.join(DATA_DIR, "water_type_needs_review.csv")

# %% Load
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
print(f"Loaded {len(df):,} rows from {INPUT_CSV}")

# %% Basic hygiene
required_cols = [
    'id','page_number','inferred_latitude','inferred_longitude',
    'actual_latitude','actual_longitude','location','townships_ranges_sections',
    'watersource_name','actual_county','inferred_county','dates_of_recording',
    'temporal_resolution','units_of_measurement','water_type','keyterms'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected column(s): {missing}")

df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# %% Preview helper
def preview(dataframe, n=50, random=False, start=None, end=None):
    if random:
        return dataframe.sample(n).style
    elif start is not None and end is not None:
        return dataframe.loc[start:end].style
    else:
        return dataframe.head(n).style

# %% Unique key
def make_key(row: pd.Series) -> str:
    return f"{row['id']}_{row['page_number']}"

df['id_page_number'] = df.apply(make_key, axis=1)

# %% Coordinates

def _to_float_or_none(v: Any) -> Optional[float]:
    try:
        if v == '' or v is None:
            return None
        return float(str(v))
    except Exception:
        return None

actual_lat = df['actual_latitude'].map(_to_float_or_none)
actual_lon = df['actual_longitude'].map(_to_float_or_none)
inf_lat    = df['inferred_latitude'].map(_to_float_or_none)
inf_lon    = df['inferred_longitude'].map(_to_float_or_none)

best_lat = actual_lat.where(actual_lat.notna(), inf_lat)
best_lon = actual_lon.where(actual_lon.notna(), inf_lon)

df['latitude']  = best_lat
df['longitude'] = best_lon

# %% Water type categorization
categories = {
    'groundwater': 'Groundwater',
    'stream discharge': 'Stream discharge',
    'precipitation': 'Precipitation',
    'spring': 'Springs',
    'reservoir': 'Reservoir',
    'water quality': 'Water quality',
    'irrigation': 'Irrigation',
    'not water related': 'Not water related',
    'other': 'Other'
}

def classify_water_type(raw: str) -> Tuple[Optional[str], Optional[str]]:
    if not raw or pd.isna(raw):
        return None, 'MISSING'
    s = raw.lower().strip()
    for key, label in categories.items():
        if key in s:
            return label, None
    return None, raw

results = df['water_type'].apply(classify_water_type)
df['water_type_clean'] = results.apply(lambda x: x[0])
df['water_type_review'] = results.apply(lambda x: x[1])

# Export water_type_review rows if any
needs_wt_review = df[df['water_type_review'].notna()]
if not needs_wt_review.empty:
    try:
        needs_wt_review.to_csv(WATER_TYPE_REVIEW_CSV, index=False)
        print(f"Exported {len(needs_wt_review):,} rows needing water_type review → {WATER_TYPE_REVIEW_CSV}")
    except Exception as e:
        print(f"Could not save {WATER_TYPE_REVIEW_CSV}: {e}")

# Count entries with multiple water types
multiple_types_count = 0
for raw in df['water_type'].dropna():
    s = raw.lower().strip()
    matches = sum(1 for key in categories.keys() if key in s)
    if matches > 1:
        multiple_types_count += 1

print(f"Entries with multiple water types: {multiple_types_count:,} out of {len(df[df['water_type'].notna()]):,} total entries with water_type data")

# %% Dates parsing
YEAR_MIN = 1800
YEAR_MAX = 1980

RE_YEAR4   = re.compile(r"\b(18\d{2}|19\d{2}|2000|20\d{2})\b")
RE_MMYYYY  = re.compile(r"\b(0?[1-9]|1[0-2])[\-/](18\d{2}|19\d{2}|20\d{2})\b")
RE_YYYYMMDD= re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})[\-/](0?[1-9]|1[0-2])[\-/]([0-2]?\d|3[01])\b")
RE_TWO_DIGIT_RANGE = re.compile(r"\b(18\d{2}|19\d{2})\s*[-/]\s*(\d{2})\b")

OPEN_TOKENS = (
    'present','to present','current','current year','to current year',
    'to date','t-','t/','tc','open','ongoing'
)

def _normalize_text(s: str) -> str:
    s = s.replace('\u2013','-').replace('\u2014','-')
    s = re.sub(r"[\u2212\u2012\u2015]", "-", s)
    s = s.replace(';', ',')
    s = s.replace('\u00a0', ' ')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _expand_two_digit_end(start_year: int, end_two: int) -> int:
    base_century = start_year - (start_year % 100)
    candidate = base_century + end_two
    if end_two < (start_year % 100):
        candidate += 100
    return candidate

def parse_years(raw: Any) -> Dict[str, Any]:
    out = {'year_start': None,'year_end': None,'years_list': None,
           'date_parse_status': None,'date_parse_notes': None,'needs_date_review': False}

    if raw is None:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    s = str(raw).strip()
    if s == '' or s.lower() in {'not specified','-','--','n/a','na'}:
        out['date_parse_status'] = 'UNPARSED'; out['needs_date_review'] = True; return out

    norm = _normalize_text(s)
    norm_lower = norm.lower()
    open_ended = any(tok in norm_lower for tok in OPEN_TOKENS)

    years: List[int] = []

    for y in RE_YEAR4.findall(norm):
        try: years.append(int(y))
        except: pass
    for mm, yyyy in RE_MMYYYY.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for yyyy, mm, dd in RE_YYYYMMDD.findall(norm):
        try: years.append(int(yyyy))
        except: pass
    for start, end2 in RE_TWO_DIGIT_RANGE.findall(norm):
        try:
            s4 = int(start); e2 = int(end2)
            years.append(s4); years.append(_expand_two_digit_end(s4,e2))
        except: pass

    years = [y for y in years if YEAR_MIN <= y <= YEAR_MAX]
    if not years:
        out['date_parse_status'] = 'UNPARSED'; out['date_parse_notes'] = 'no_year_<=1980_found'; out['needs_date_review'] = True; return out

    years = sorted(set(years))
    out['years_list'] = years
    out['year_start'] = min(years)
    if open_ended:
        out['year_end'] = None; out['date_parse_status'] = 'OPEN_ENDED_IGNORED'; out['needs_date_review'] = True
    else:
        out['year_end'] = max(years)
        if len(years) == 1: out['date_parse_status'] = 'SINGLE_YEAR'
        elif len(years) == 2 and out['year_end'] != out['year_start']: out['date_parse_status'] = 'EXACT_RANGE'
        else: out['date_parse_status'] = 'MULTI_YEARS_OR_RANGES'
    return out

# %% Apply parser
df.rename(columns={'dates_of_recording': 'dates_of_recording_raw'}, inplace=True)
parsed = df['dates_of_recording_raw'].apply(parse_years).apply(pd.Series)
for col in parsed.columns:
    df[col] = parsed[col]

df['year_start'] = df['year_start'].astype('Int64')
df['year_end']   = df['year_end'].astype('Int64')

n_unparsed = (df['date_parse_status'] == 'UNPARSED').sum()
print(f"\nRows with no parsable year: {n_unparsed:,}")

# %% Date parsing failure analysis
total_rows = len(df)
empty_or_na = df['dates_of_recording_raw'].isna().sum()
empty_strings = (df['dates_of_recording_raw'].astype(str).str.strip() == '').sum()
standard_na_values = df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']).sum()

# Total "missing" data (empty/NA/standard missing values)
total_missing = len(df[(df['dates_of_recording_raw'].isna()) | 
                      (df['dates_of_recording_raw'].astype(str).str.strip() == '') |
                      (df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

# Unparsed with actual content
unparsed_with_content = len(df[(df['date_parse_status'] == 'UNPARSED') & 
                              (~df['dates_of_recording_raw'].isna()) &
                              (df['dates_of_recording_raw'].astype(str).str.strip() != '') &
                              (~df['dates_of_recording_raw'].astype(str).str.lower().isin(['not specified', '-', '--', 'n/a', 'na']))])

print(f"\nDate parsing breakdown:")
print(f"Total rows: {total_rows:,}")
print(f"Empty/NA dates: {empty_or_na:,}")
print(f"Empty strings: {empty_strings:,}")
print(f"Standard NA values ('not specified', '-', 'n/a', etc.): {standard_na_values:,}")
print(f"Total missing data: {total_missing:,}")
print(f"Unparsed but has content: {unparsed_with_content:,}")
print(f"Successfully parsed: {total_rows - n_unparsed:,}")

In [ ]:
# %% Quick sanity checks
print("\nSample of parsed years:")
print(df[['dates_of_recording_raw','year_start','year_end','date_parse_status']].head(12))
print("\nUnique water_type_clean values:")
print(df['water_type_clean'].dropna().unique()[:50])
print("\nRows needing water_type_review:")
print(df[df['water_type_review'].notna()][['water_type','water_type_review']].head(20))

# %% Summaries
# Exclude rows where water_type_clean is missing
df_summary = df[df['water_type_clean'].notna()].copy()

# By water type only
summary_by_type = (
    df_summary
    .groupby('water_type_clean')
    .agg(
        n_rows=('id_page_number','size'),
        n_pages=('id_page_number','nunique'),
        n_docs=('id','nunique')
    )
    .reset_index()
)
print("\nSummary by water type:")
print(summary_by_type.head(20))

# By water type × year
summary_by_year = (
    df_summary
    .groupby(['water_type_clean','year_start'])
    .agg(
        n_rows=('id_page_number','size'),
        n_pages=('id_page_number','nunique'),
        n_docs=('id','nunique')
    )
    .reset_index()
    .sort_values(['water_type_clean','year_start'])
)
print("\nSummary by water type × year:")
print(summary_by_year.head(20))

# By water type × decade
df_summary['decade'] = (df_summary['year_start'] // 10) * 10
summary_by_decade = (
    df_summary
    .groupby(['water_type_clean','decade'])
    .agg(
        n_rows=('id_page_number','size'),
        n_pages=('id_page_number','nunique'),
        n_docs=('id','nunique')
    )
    .reset_index()
    .sort_values(['water_type_clean','decade'])
)
print("\nSummary by water type × decade:")
print(summary_by_decade.head(20))


In [ ]:
# --- Discover DataFrames in memory and save summaries + full table ---
import pandas as pd
from artifacts_io import save_artifacts

# 1) Collect all pandas DataFrames alive in the current notebook
dfs_in_mem = {name: obj for name, obj in globals().items() if isinstance(obj, pd.DataFrame)}

if not dfs_in_mem:
    raise RuntimeError("No pandas DataFrames found in memory. Run the upstream analysis cells first.")

print("Found DataFrames:")
for k, v in sorted(dfs_in_mem.items()):
    print(f"  - {k:28s} shape={v.shape} cols={list(v.columns)[:6]}{'...' if v.shape[1]>6 else ''}")

# 2) Heuristics for "summary" tables: common naming patterns & presence of groupby-like columns
name_hits = ("summary", "sum_", "_sum", "agg", "group", "pivot", "tbl", "by_", "_by")
summary_like = [k for k in dfs_in_mem if any(tok in k.lower() for tok in name_hits)]

# If nothing matched by name, look for “tall-to-wide” hints (few rows, many categorical columns)
if not summary_like:
    for k, df in dfs_in_mem.items():
        if df.shape[0] < max(200, 0.05 * max(d.shape[0] for d in dfs_in_mem.values())):
            summary_like.append(k)

# 3) Heuristic for "everything" table: largest number of rows
everything_key = max(dfs_in_mem, key=lambda k: dfs_in_mem[k].shape[0])

print("\nAuto-identified candidates:")
print("  Summary-like tables:", summary_like if summary_like else "(none found by heuristics)")
print("  Everything table    :", everything_key, dfs_in_mem[everything_key].shape)

# 4) OPTIONAL: override explicitly if you know the exact names
# summary_like = ['df_summary_events','df_summary_sites']   # <-- edit me if desired
# everything_key = 'df_all_clean'                           # <-- edit me if desired

# Build what to save
dfs_to_save = {}
for key in summary_like:
    dfs_to_save[key] = dfs_in_mem[key]
dfs_to_save[everything_key] = dfs_in_mem[everything_key]   # ensure the full table is included

meta = {
    "notes": "Summaries + full analysis table exported for plotting/other notebooks",
    "source_notebook": "working_datacrunch.ipynb",
    "summary_keys": summary_like,
    "everything_key": everything_key,
}

run_dir = save_artifacts(dfs_to_save, meta=meta, out_dir="artifacts")
print("\nSaved artifacts to:", run_dir)

